# Cartographie prédictive GWS — Lasso

Notebook Google Colab généré à partir du script Python d'origine : **`Lasso.py`**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Chaque cellule exécute directement son étape, comme le script d'origine (qui n'a pas de fonction `main()`).
Les résultats sont écrits dans `input_data_ML/GWS_PREDICTIONS_GEE_LASSO` sur Drive.

Le code du script est repris tel quel. Les seules lignes ajoutées ou modifiées pour Colab
sont repérées par le commentaire `# [COLAB]`.

### Description (en-tête du script d'origine)

```text
# ============================================================
#  CARTOGRAPHIE PREDICTIVE GWS
#  (TRAIN = longue période, TEST = 5% des dates, CLIP GEE, SANS SOUS-ÉCHANTILLONNAGE)
#
#  - Modèle Lasso (sklearn) sur grille GLDAS (~25 km)
#
#  - Variables explicatives (via name_files.txt) :
#       * Precipitations_IDW        (journalier)   [prioritaire]
#         ou Precipitations_API_IDW (journalier)   [fallback]
#         ou CHIRPS_Clipped         (journalier)   [fallback ultime]
#       * MODIS_LST_Clipped         (journalier)
#       * MODIS_NDVI_Clipped        (16 jours, étendu "prev-date")
#       * MODIS_ET_Clipped          (~8 jours, étendu "prev-date")
#       * ERA5_Qsb_Clipped          (journalier)
#       * SMAP_RZSM_Clipped         (journalier)
#       * SMAP_SoilM_Clipped        (journalier)
#
#  - Cible :
#       * GLDAS_GWS_Clipped (ou équivalent)
#
#  - Features finales :
#       [PRECIP, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, sinDOY, cosDOY]
#
#  - Rééchantillonnage NoData OK
#  - Remplissage NaN d'entrée → cartes SANS TROUS (dans Regions)
#  - Regions reconstruit depuis les rasters (pas besoin d'asset GEE)
#
#  ➤ Périodes :
#    Pool TRAIN/TEST : 2021-01-01 → 2025-06-30
#    TEST = 5% des dates (aléatoire)
#    FORECAST : 2025-07-01 → 2025-10-30
# ============================================================
```

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab (scikit-learn, numpy et joblib le sont).

In [ ]:
!pip install -q rasterio

### A.3 Imports

In [ ]:
import os
import re
import time
import bisect
from datetime import datetime, date

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

# [COLAB] Versions des bibliothèques utilisées
from importlib.metadata import version as _pkg_version
for _pkg in ('numpy', 'scikit-learn', 'rasterio'):
    print(f"{_pkg:13s}: {_pkg_version(_pkg)}")

## 1. PARAMÈTRES GÉNÉRAUX

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# ------------------------------------------------------------
# 1) PARAMÈTRES GÉNÉRAUX
# ------------------------------------------------------------
np.random.seed(42)

# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR = "/content/drive/MyDrive/input_data_ML"
NAME_FILE = os.path.join(BASE_DIR, "name_files.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon le dossier de sortie serait créé hors de Drive et perdu en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

OUT_DIR = os.path.join(BASE_DIR, "GWS_PREDICTIONS_GEE_LASSO")
os.makedirs(OUT_DIR, exist_ok=True)

NODATA_VALUE = -9999.0
TEST_FRACTION = 0.05

# Fenêtres temporelles
TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

FORECAST_START = datetime(2025, 7, 1)
FORECAST_END   = datetime(2025, 10, 30)

# Dates exclues (LST manquante 2022-10-11 → 2022-10-22)
EXCLUDED_DATES = {date(2022, 10, d) for d in range(11, 23)}

print("=== PARAMÈTRES GÉNÉRAUX ===")
print(" BASE_DIR :", BASE_DIR)
print(" NAME_FILE :", NAME_FILE)
print(" OUT_DIR :", OUT_DIR)
print(" TRAIN_START/END :", TRAIN_START, "→", TRAIN_END)
print(" TEST_FRACTION :", TEST_FRACTION)
print(" FORECAST_START/END :", FORECAST_START.date(), "→", FORECAST_END.date())
print("============================\n")

print("Dates exclues (LST manquante) :")
for d in sorted(EXCLUDED_DATES):
    print(" -", d)

### 1.1 SECTIONS (fallback)

In [ ]:
# ------------------------------------------------------------
# 1.1) SECTIONS (fallback)
# ------------------------------------------------------------
SECTION_PRECIP = (
    "Precipitations_IDW",
    "Precipitations_API_IDW",
    "CHIRPS_Clipped",
    "Precipitations_API",
)

SECTION_LST   = ("MODIS_LST_Clipped",)
SECTION_NDVI  = ("MODIS_NDVI_Clipped",)
SECTION_ET    = ("MODIS_ET_Clipped",)
SECTION_GWS   = ("GLDAS_GWS_Clipped", "Total_GWS", "GLDAS_GWS")
SECTION_QSB   = ("ERA5_Qsb_Clipped",)
SECTION_RZSM  = ("SMAP_RZSM_Clipped",)
SECTION_SOILM = ("SMAP_SoilM_Clipped",)

## 2. FONCTIONS UTILITAIRES

Adaptation Colab : les entrées de `name_files.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# ------------------------------------------------------------
# 2) FONCTIONS UTILITAIRES
# ------------------------------------------------------------
def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p


def parse_name_file(txt_path):
    """Parse name_files.txt -> dict {SectionName: [relative/path.tif, ...]}."""
    sections = {}
    current = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
            else:
                if current is None:
                    continue
                sections[current].append(_win_to_colab_path(line))  # [COLAB] chemin Windows -> Colab
    return sections


def pick_section(sections_dict, candidates):
    """Retourne le premier nom de section existant, parmi candidates."""
    for name in candidates:
        if name in sections_dict:
            return name
    raise RuntimeError(
        f"Aucune section trouvée parmi {candidates}. "
        f"Sections disponibles: {list(sections_dict.keys())}"
    )


def extract_date_from_filename(path):
    """
    Extrait une date depuis le nom du fichier.
    Supporte:
      - YYYYMMDD
      - DDMMYYYY
    """
    name = os.path.basename(path)
    m = re.search(r"(\d{8})", name)
    if not m:
        raise ValueError(f"Impossible d'extraire la date depuis : {name}")
    s = m.group(1)

    yyyy = int(s[:4])
    if 1900 <= yyyy <= 2100:
        return datetime.strptime(s, "%Y%m%d").date()

    dd = int(s[:2]); mm = int(s[2:4]); yyyy = int(s[4:])
    if 1 <= dd <= 31 and 1 <= mm <= 12 and 1900 <= yyyy <= 2100:
        return datetime.strptime(s, "%d%m%Y").date()

    return datetime.strptime(s, "%Y%m%d").date()


def build_date_to_path_dict_from_section(sections, section_name):
    """Construit dict {date: abs_path} pour une section donnée."""
    d = {}
    for rel_path in sections[section_name]:
        abs_path = os.path.normpath(os.path.join(BASE_DIR, rel_path))
        if not os.path.isfile(abs_path):
            print(f"⚠️ Fichier introuvable (ignoré) : {abs_path}")
            continue
        try:
            date_key = extract_date_from_filename(abs_path)
            d[date_key] = abs_path
        except ValueError as e:
            print("⚠️", e)
    return d


def read_raster_raw(path):
    """Lit un raster (bande 1) en float32, sans toucher NoData."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
    return arr, profile, nodata


def to_float_with_nan(arr, nodata):
    """Convertit en float32 et remplace nodata -> NaN."""
    arr = arr.astype("float32")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr


def reproject_to_template(
    src_arr,
    src_profile,
    template_profile,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average,
):
    """Rééchantillonne src_arr sur la grille template_profile."""
    dst_arr = np.full(
        (template_profile["height"], template_profile["width"]),
        dst_nodata,
        dtype="float32",
    )
    kwargs = dict(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        dst_transform=template_profile["transform"],
        dst_crs=template_profile["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    if src_nodata is not None:
        kwargs["src_nodata"] = src_nodata
    reproject(**kwargs)
    return dst_arr


def fill_nan_with_mean(arr):
    """Remplace les NaN par la moyenne du raster."""
    if np.all(np.isnan(arr)):
        return arr
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr)


def build_prev_date_support(date_dict):
    """Support 'dernière date <= target_date' (NDVI/ET)."""
    dates_sorted = sorted(date_dict.keys())

    def get_prev(target_date):
        if not dates_sorted:
            return None
        idx = bisect.bisect_right(dates_sorted, target_date) - 1
        if idx < 0:
            return None
        return dates_sorted[idx]

    return get_prev

## 3. REGION MASK (Regions reconstruit)

In [ ]:
# ------------------------------------------------------------
# 3) REGION MASK (Regions reconstruit)
# ------------------------------------------------------------
REGION_MASK = None

def update_region_mask(precip_res, lst_res, ndvi_res, template_profile):
    global REGION_MASK
    base_mask = ~(np.isnan(precip_res) & np.isnan(lst_res) & np.isnan(ndvi_res))
    if REGION_MASK is None:
        REGION_MASK = base_mask
        print("\n🗺️ Masque 'Regions' initial construit à partir des rasters (1er jour).")
    else:
        REGION_MASK = REGION_MASK | base_mask

def get_region_mask(template_profile):
    global REGION_MASK
    if REGION_MASK is None:
        print("⚠️ REGION_MASK non initialisé, on prend toute la grille.")
        REGION_MASK = np.ones((template_profile["height"], template_profile["width"]), dtype=bool)
    return REGION_MASK

### Contrôle des entrées de `name_files.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichier présent sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _missing = [_e for _e in _entries
                if not os.path.isfile(os.path.normpath(os.path.join(BASE_DIR, _e)))]
    _n_ok = len(_entries) - len(_missing)
    _flag = "✅" if (_n_ok > 0 and not _missing) else ("⚠️" if _n_ok > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_ok} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {os.path.normpath(os.path.join(BASE_DIR, _e))}")

## 4. INDEXATION DES FICHIERS (via name_files.txt)

In [ ]:
# ------------------------------------------------------------
# 4) INDEXATION DES FICHIERS (via name_files.txt)
# ------------------------------------------------------------
print("\n=== INDEXATION DES FICHIERS PAR DATE (via name_files.txt) ===")
sections = parse_name_file(NAME_FILE)

sec_precip = pick_section(sections, SECTION_PRECIP)
sec_lst    = pick_section(sections, SECTION_LST)
sec_ndvi   = pick_section(sections, SECTION_NDVI)
sec_et     = pick_section(sections, SECTION_ET)
sec_gws    = pick_section(sections, SECTION_GWS)
sec_qsb    = pick_section(sections, SECTION_QSB)
sec_rzsm   = pick_section(sections, SECTION_RZSM)
sec_soilm  = pick_section(sections, SECTION_SOILM)

print("Sections utilisées :")
print(" - PRECIP:", sec_precip)
print(" - LST   :", sec_lst)
print(" - NDVI  :", sec_ndvi)
print(" - ET    :", sec_et)
print(" - GWS   :", sec_gws)
print(" - Qsb   :", sec_qsb)
print(" - RZSM  :", sec_rzsm)
print(" - SoilM :", sec_soilm)

precip_dict = build_date_to_path_dict_from_section(sections, sec_precip)
lst_dict    = build_date_to_path_dict_from_section(sections, sec_lst)
ndvi_dict   = build_date_to_path_dict_from_section(sections, sec_ndvi)
et_dict     = build_date_to_path_dict_from_section(sections, sec_et)
gws_dict    = build_date_to_path_dict_from_section(sections, sec_gws)
qsb_dict    = build_date_to_path_dict_from_section(sections, sec_qsb)
rzsm_dict   = build_date_to_path_dict_from_section(sections, sec_rzsm)
soilm_dict  = build_date_to_path_dict_from_section(sections, sec_soilm)

print("\nNb dates PRECIP :", len(precip_dict))
print("Nb dates LST    :", len(lst_dict))
print("Nb dates NDVI   :", len(ndvi_dict))
print("Nb dates ET     :", len(et_dict))
print("Nb dates GWS    :", len(gws_dict))
print("Nb dates Qsb    :", len(qsb_dict))
print("Nb dates RZSM   :", len(rzsm_dict))
print("Nb dates SoilM  :", len(soilm_dict))

get_ndvi_date_for = build_prev_date_support(ndvi_dict)
get_et_date_for   = build_prev_date_support(et_dict)

## 5. DATES TRAIN / TEST

In [ ]:
# ------------------------------------------------------------
# 5) DATES TRAIN / TEST
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DES DATES TRAIN / TEST (5% dates TEST) ===")

all_common_dates = sorted(
    set(precip_dict.keys())
    & set(lst_dict.keys())
    & set(gws_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

print("Nombre total de dates communes :", len(all_common_dates))
if not all_common_dates:
    raise RuntimeError("⚠️ Aucune date commune entre les variables journalières.")

date_candidates = [
    d for d in all_common_dates
    if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)
]
if not date_candidates:
    raise RuntimeError("⚠️ Aucune date candidate dans la fenêtre TRAIN (après exclusion).")

train_dates, test_dates = train_test_split(
    date_candidates,
    test_size=TEST_FRACTION,
    random_state=42,
    shuffle=True,
)
train_dates = sorted(train_dates)
test_dates  = sorted(test_dates)

print(f"Train dates : {len(train_dates)} | {min(train_dates)} → {max(train_dates)}")
print(f"Test  dates : {len(test_dates)}  | {min(test_dates)} → {max(test_dates)}")

## 6. BUILD DATASET (pixels)

In [ ]:
# ------------------------------------------------------------
# 6) BUILD DATASET (pixels)
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DU DATASET (pixels TRAIN + TEST) ===")

X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []
template_profile = None

def build_Xy_for_date(date_key, template_profile, role="train"):
    ndvi_date = get_ndvi_date_for(date_key)
    if ndvi_date is None:
        print(f"⏭️ [{role}] {date_key} : pas de NDVI <= date, ignoré.")
        return None, None, template_profile

    et_date = get_et_date_for(date_key)
    if et_date is None:
        print(f"⏭️ [{role}] {date_key} : pas de ET <= date, ignoré.")
        return None, None, template_profile

    if date_key not in gws_dict:
        print(f"⏭️ [{role}] {date_key} : GWS manquant, ignoré.")
        return None, None, template_profile

    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    if template_profile is None:
        template_profile = gws_prof
        print("\nGrille template (GWS/GLDAS) :")
        print(" - CRS  :", template_profile["crs"])
        print(" - Taille :", template_profile["width"], "x", template_profile["height"])
        print(" - Résolution approx (deg) :", template_profile["transform"][0])

    # Vérif journalières
    needed = (precip_dict, lst_dict, qsb_dict, rzsm_dict, soilm_dict)
    if any(date_key not in dct for dct in needed):
        print(f"⏭️ [{role}] {date_key} : variable journalière manquante, ignoré.")
        return None, None, template_profile

    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata,    dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata,   dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata,     dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata,    dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata,   dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata,  dst_nodata=np.nan)

    update_region_mask(precip_res, lst_res, ndvi_res, template_profile)
    region_mask = get_region_mask(template_profile)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    mask_valid = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_valid):
        print(f"⏭️ [{role}] {date_key} : aucun pixel valide, ignoré.")
        return None, None, template_profile

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels = np.count_nonzero(mask_valid)
    sin_plane = np.full(n_pixels, sin_doy, dtype="float32")
    cos_plane = np.full(n_pixels, cos_doy, dtype="float32")

    X_day = np.stack(
        [
            precip_f[mask_valid],
            lst_f[mask_valid],
            ndvi_f[mask_valid],
            et_f[mask_valid],
            qsb_f[mask_valid],
            rzsm_f[mask_valid],
            soilm_f[mask_valid],
            sin_plane,
            cos_plane,
        ],
        axis=1,
    )
    y_day = gws_arr[mask_valid]

    row_mask = (~np.isnan(y_day)) & (~np.isnan(X_day).any(axis=1))
    if not np.any(row_mask):
        print(f"⏭️ [{role}] {date_key} : pixels invalides (NaN), ignoré.")
        return None, None, template_profile

    X_day = X_day[row_mask]
    y_day = y_day[row_mask]

    print(f" ✔️ [{role}] {date_key} : {X_day.shape[0]} pixels utilisés.")
    return X_day, y_day, template_profile


for d in train_dates:
    X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="train")
    if X_day is None:
        continue
    X_train_list.append(X_day)
    y_train_list.append(y_day)

for d in test_dates:
    X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="test")
    if X_day is None:
        continue
    X_test_list.append(X_day)
    y_test_list.append(y_day)

if not X_train_list:
    raise RuntimeError("⚠️ Aucun pixel d'entraînement valide trouvé.")
if not X_test_list:
    raise RuntimeError("⚠️ Aucun pixel de test valide trouvé.")

X_train = np.vstack(X_train_list).astype("float32")
y_train = np.concatenate(y_train_list).astype("float32")
X_test  = np.vstack(X_test_list).astype("float32")
y_test  = np.concatenate(y_test_list).astype("float32")

print(f"\nPixels TRAIN : {X_train.shape[0]}")
print(f"Pixels TEST  : {X_test.shape[0]}")
print("Exemple feature :", X_train[0])

## 7. ENTRAINEMENT LASSO

In [ ]:
# ------------------------------------------------------------
# 7) ENTRAINEMENT LASSO
# ------------------------------------------------------------
print("\n=== ENTRAÎNEMENT LASSO (avec StandardScaler) ===")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, shuffle=True
)
print(f"Taille X_tr : {X_tr.shape}, X_val : {X_val.shape}")

# ✅ Lasso a besoin d’un scaling (sinon alpha dépend trop des unités)
model = Pipeline(
    steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("lasso", Lasso(alpha=0.001, max_iter=20000, random_state=42)),
    ]
)

t0 = time.time()
model.fit(X_tr, y_tr)
t1 = time.time()
training_time_ms = (t1 - t0) * 1000.0

y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

r2_train   = r2_score(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train  = mean_absolute_error(y_train, y_train_pred)

r2_test   = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test  = mean_absolute_error(y_test, y_test_pred)

print("\n╔═════════ MODÈLE LASSO ═════════╗")
print(f" ➤ Temps d'entraînement : {training_time_ms:.2f} ms")
print(f" ➤ R² TRAIN : {r2_train:.4f}")
print(f" ➤ MAE TRAIN : {mae_train:.4f}")
print(f" ➤ RMSE TRAIN: {rmse_train:.4f}")
print("----------------------------------")
print(f" ➤ R² TEST : {r2_test:.4f}")
print(f" ➤ MAE TEST : {mae_test:.4f}")
print(f" ➤ RMSE TEST: {rmse_test:.4f}")
print("╚════════════════════════════════╝")

model_path = os.path.join(OUT_DIR, "Lasso_GWS_model.joblib")
joblib.dump(model, model_path)
print(f"\n💾 Modèle sauvegardé : {model_path}")

## 8. EVALUATION CARTOGRAPHIQUE (dates TEST)

In [ ]:
# ------------------------------------------------------------
# 8) EVALUATION CARTOGRAPHIQUE (dates TEST)
# ------------------------------------------------------------
print("\n=== ÉVALUATION CARTOGRAPHIQUE (DATES TEST) ===")
region_mask = get_region_mask(template_profile)

all_true, all_pred = [], []

for date_key in test_dates:
    print(f"\n🧪 Test cartographique : {date_key}")

    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None or date_key not in gws_dict:
        print(f" ⏭️ {date_key} : NDVI/ET/GWS indisponible, ignorée.")
        continue

    needed = (precip_dict, lst_dict, qsb_dict, rzsm_dict, soilm_dict)
    if any(date_key not in dct for dct in needed):
        print(f" ⏭️ {date_key} : variable journalière manquante, ignorée.")
        continue

    # GWS vérité terrain
    gws_raw, _, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    # Entrées
    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata,    dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata,   dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata,     dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata,    dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata,   dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata,  dst_nodata=np.nan)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels_full = precip_f.size
    sin_plane_full = np.full(n_pixels_full, sin_doy, dtype="float32")
    cos_plane_full = np.full(n_pixels_full, cos_doy, dtype="float32")

    X_all = np.stack(
        [
            precip_f.ravel(),
            lst_f.ravel(),
            ndvi_f.ravel(),
            et_f.ravel(),
            qsb_f.ravel(),
            rzsm_f.ravel(),
            soilm_f.ravel(),
            sin_plane_full,
            cos_plane_full,
        ],
        axis=1,
    )

    if np.isnan(X_all).any():
        print(f" ⏭️ {date_key} : NaN dans X_all, ignorée.")
        continue

    y_pred_all = model.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    mask_gws = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_gws):
        print(" ⚠️ Aucun pixel valide dans GWS pour cette date.")
        continue

    y_true = gws_arr[mask_gws]
    y_pred = pred_clip[mask_gws]

    r2_d   = r2_score(y_true, y_pred)
    mae_d  = mean_absolute_error(y_true, y_pred)
    rmse_d = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f" ➤ R²  : {r2_d:.4f}")
    print(f" ➤ MAE : {mae_d:.4f}")
    print(f" ➤ RMSE: {rmse_d:.4f}")

    all_true.append(y_true)
    all_pred.append(y_pred)

    out_profile_test = template_profile.copy()
    out_profile_test.update(dtype="float32", count=1, nodata=NODATA_VALUE)

    out_name_test = os.path.join(OUT_DIR, f"GWS_TEST_PRED_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name_test, "w", **out_profile_test) as dst:
        dst.write(pred_clip, 1)
    print(f" ✔️ Carte test : {out_name_test}")

if all_true:
    all_true_flat = np.concatenate(all_true)
    all_pred_flat = np.concatenate(all_pred)
    r2_global   = r2_score(all_true_flat, all_pred_flat)
    mae_global  = mean_absolute_error(all_true_flat, all_pred_flat)
    rmse_global = np.sqrt(mean_squared_error(all_true_flat, all_pred_flat))
    print("\n📊 MÉTRIQUES GLOBALES (TEST)")
    print(f" ➤ R² global : {r2_global:.4f}")
    print(f" ➤ MAE global : {mae_global:.4f}")
    print(f" ➤ RMSE global: {rmse_global:.4f}")
else:
    print("\n⚠️ Aucune métrique globale (aucun pixel valide).")

## 9. FORECAST (prédictions cartographiques)

In [ ]:
# ------------------------------------------------------------
# 9) FORECAST (prédictions cartographiques)
# ------------------------------------------------------------
print("\n=== PHASE DE PRÉDICTION CARTOGRAPHIQUE (FORECAST) ===")

out_profile = template_profile.copy()
out_profile.update(dtype="float32", count=1, nodata=NODATA_VALUE)

all_forecast_dates = sorted(
    set(precip_dict.keys())
    & set(lst_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

if not all_forecast_dates:
    print("⚠️ Aucune date commune (PRECIP ∩ LST ∩ Qsb ∩ RZSM ∩ SoilM). Stop.")
else:
    print("Plage possible :", min(all_forecast_dates), "→", max(all_forecast_dates))

user_start = FORECAST_START.date()
user_end   = FORECAST_END.date()

forecast_start_date = max(user_start, min(all_forecast_dates)) if all_forecast_dates else user_start
forecast_end_date   = min(user_end,   max(all_forecast_dates)) if all_forecast_dates else user_end

print("Fenêtre demandée :", user_start, "→", user_end)
print("Fenêtre utilisée :", forecast_start_date, "→", forecast_end_date)

forecast_dates = [
    d for d in all_forecast_dates
    if (forecast_start_date <= d <= forecast_end_date) and (d not in EXCLUDED_DATES)
]

print(f"Nombre de dates éligibles : {len(forecast_dates)}")
if forecast_dates:
    print("Première :", min(forecast_dates))
    print("Dernière :", max(forecast_dates))

region_mask = get_region_mask(template_profile)

for date_key in forecast_dates:
    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(f"⏭️ {date_key} : NDVI/ET indisponible, ignorée.")
        continue

    needed = (precip_dict, lst_dict, qsb_dict, rzsm_dict, soilm_dict)
    if any(date_key not in dct for dct in needed):
        print(f"⏭️ {date_key} : variable journalière manquante, ignorée.")
        continue

    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata,    dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata,   dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata,     dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata,    dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata,   dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata,  dst_nodata=np.nan)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels_full = precip_f.size
    sin_plane_full = np.full(n_pixels_full, sin_doy, dtype="float32")
    cos_plane_full = np.full(n_pixels_full, cos_doy, dtype="float32")

    X_all = np.stack(
        [
            precip_f.ravel(),
            lst_f.ravel(),
            ndvi_f.ravel(),
            et_f.ravel(),
            qsb_f.ravel(),
            rzsm_f.ravel(),
            soilm_f.ravel(),
            sin_plane_full,
            cos_plane_full,
        ],
        axis=1,
    )

    if np.isnan(X_all).any():
        print(f"⏭️ {date_key} : NaN dans X_all, ignorée.")
        continue

    y_pred_all = model.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    out_name = os.path.join(OUT_DIR, f"GWS_PRED_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name, "w", **out_profile) as dst:
        dst.write(pred_clip, 1)

    print(f" ✔️ Carte prédite : {out_name}")

# Récap
if os.path.isdir(OUT_DIR):
    tif_files = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".tif")]
    print(f"\n📂 Total GeoTIFF générés dans {OUT_DIR} : {len(tif_files)}")
else:
    print(f"⚠️ OUT_DIR introuvable : {OUT_DIR}")

## Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers présents dans le dossier de sortie, par extension
from collections import Counter

_count = Counter(os.path.splitext(_f)[1].lower() or "(sans extension)" for _f in os.listdir(OUT_DIR))
for _ext, _n in sorted(_count.items()):
    print(f"{_n:6d} fichier(s) {_ext}")
print("Dossier de sortie :", OUT_DIR)